# 🪨 USCS Soil Classification Tool
**Based on ASTM D2487 — Unified Soil Classification System**

This tool classifies soils as fine-grained (silts & clays) or coarse-grained (sands & gravels) using standard geotechnical lab test inputs.

**How to use:**
1. Run Cell 1 to install/import all libraries
2. Run Cell 2 to load the classification logic
3. Run Cell 3 to launch the interactive tool — adjust sliders and click **Classify**

In [ ]:
# ── Cell 1: Install & Import Libraries ───────────────────────────────────────
# Run this cell first. If ipywidgets isn't installed, uncomment the line below.
# !pip install ipywidgets matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline
print('✓ Libraries loaded successfully')

In [ ]:
# ── Cell 2: Classification Logic & Plotting Functions ────────────────────────
# Run this cell to load all the functions before using the tool.

# ── USCS Classification ───────────────────────────────────────────────────────

def classify_fine_grained(LL, PI):
    """
    Classify fine-grained soils (>50% passing #200 sieve).
    Uses Casagrande A-line: PI = 0.73 * (LL - 20)
    """
    A_line_PI = 0.73 * (LL - 20)

    if LL < 50:
        if PI > A_line_PI and PI >= 7:
            symbol, name = 'CL', 'Lean Clay'
        elif PI < A_line_PI or PI < 4:
            symbol, name = 'ML', 'Silt of Low Plasticity'
        else:
            symbol, name = 'CL-ML', 'Silty Clay (Borderline)'
    else:
        if PI > A_line_PI:
            symbol, name = 'CH', 'Fat Clay'
        else:
            symbol, name = 'MH', 'Elastic Silt'

    organic_flag = ''
    if LL < 50 and PI < A_line_PI:
        organic_flag = '⚠  If organic odor or dark color: consider OL (Organic Silt/Clay)'
    elif LL >= 50 and PI < A_line_PI:
        organic_flag = '⚠  If organic odor or dark color: consider OH (Organic Clay)'

    return symbol, name, organic_flag


def classify_coarse_grained(pct_gravel, pct_sand, pct_fines, Cu, Cc, fines_LL=None, fines_PI=None):
    """
    Classify coarse-grained soils (<50% passing #200 sieve).
    Cu = D60/D10,  Cc = D30^2 / (D10 * D60)
    """
    if pct_gravel > pct_sand:
        base = 'G'
        well_graded = Cu >= 4 and 1 <= Cc <= 3
    else:
        base = 'S'
        well_graded = Cu >= 6 and 1 <= Cc <= 3

    if pct_fines < 5:
        suffix = 'W' if well_graded else 'P'
        fines_type = ''
    elif pct_fines > 12:
        if fines_LL is None or fines_PI is None:
            return None, 'Atterberg limits required for soils with >12% fines', ''
        A_line_PI = 0.73 * (fines_LL - 20)
        if fines_PI >= 7 and fines_PI > A_line_PI:
            suffix, fines_type = 'C', 'Clayey'
        else:
            suffix, fines_type = 'M', 'Silty'
    else:
        w_p = 'W' if well_graded else 'P'
        if fines_LL is None or fines_PI is None:
            suffix = f'{w_p}-?M or {w_p}-?C'
            fines_type = '(Atterberg limits needed for dual symbol)'
        else:
            A_line_PI = 0.73 * (fines_LL - 20)
            f_suffix = 'C' if (fines_PI >= 7 and fines_PI > A_line_PI) else 'M'
            suffix = f'{w_p}-{base}{f_suffix}'
            fines_type = 'Dual symbol'

    symbol = f'{base}{suffix}'
    name_map = {
        'GW': 'Well-Graded Gravel', 'GP': 'Poorly-Graded Gravel',
        'GM': 'Silty Gravel',       'GC': 'Clayey Gravel',
        'SW': 'Well-Graded Sand',   'SP': 'Poorly-Graded Sand',
        'SM': 'Silty Sand',         'SC': 'Clayey Sand',
    }
    name = name_map.get(symbol, f"{fines_type} {'Gravel' if base == 'G' else 'Sand'} ({symbol})")
    return symbol, name, ''


# ── Plotting ──────────────────────────────────────────────────────────────────

def plot_casagrande(LL, PI, symbol):
    fig, ax = plt.subplots(figsize=(9, 6))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    LL_range = np.linspace(0, 110, 300)
    A_line = np.maximum(0, 0.73 * (LL_range - 20))
    U_line = np.maximum(0, 0.90 * (LL_range - 8))

    ax.plot(LL_range, A_line, color='#e94560', linewidth=2,   label='A-line: PI = 0.73(LL−20)')
    ax.plot(LL_range, U_line, color='#f5a623', linewidth=1.5, linestyle='--', label='U-line: PI = 0.90(LL−8)')
    ax.axvline(x=50, color='#a8dadc', linewidth=1.2, linestyle=':', label='LL = 50')
    ax.axhline(y=7,  color='#a8dadc', linewidth=1.2, linestyle=':', label='PI = 7')

    zone_labels = [
        (65, 35, 'CH\nFat Clay'),
        (80, 8,  'MH\nElastic Silt'),
        (30, 14, 'CL\nLean Clay'),
        (25, 3,  'ML / CL-ML'),
        (80, 25, 'CH / OH'),
    ]
    for lx, py, txt in zone_labels:
        ax.text(lx, py, txt, color='#ccc', fontsize=8, alpha=0.75, ha='center')

    ax.scatter([LL], [PI], color='#00ff88', s=150, zorder=5, label=f'Your sample: {symbol}')
    ax.annotate(f'  {symbol}', (LL, PI), color='#00ff88', fontsize=12, fontweight='bold')

    ax.set_xlim(0, 110)
    ax.set_ylim(0, 70)
    ax.set_xlabel('Liquid Limit (LL)', color='white', fontsize=11)
    ax.set_ylabel('Plasticity Index (PI)', color='white', fontsize=11)
    ax.set_title('Casagrande Plasticity Chart — USCS Classification', color='white', fontsize=13, pad=15)
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')
    ax.legend(facecolor='#1a1a2e', edgecolor='#555', labelcolor='white', fontsize=9)
    plt.tight_layout()
    plt.savefig('casagrande_chart.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print('  Chart saved as casagrande_chart.png')


def plot_grain_size(Cu, Cc, symbol):
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    D10 = 0.1
    D60 = D10 * Cu
    D30 = np.sqrt(Cc * D10 * D60)
    diameters = np.logspace(np.log10(D10 * 0.5), np.log10(D60 * 2), 200)
    passing = 10 + 80 / (1 + np.exp(-2.5 * (np.log10(diameters) - np.log10(D30))))

    ax.semilogx(diameters, passing, color='#00ff88', linewidth=2.5, label='Grain Size Curve')
    for d, lbl in [(D10, 'D10'), (D30, 'D30'), (D60, 'D60')]:
        p = 10 + 80 / (1 + np.exp(-2.5 * (np.log10(d) - np.log10(D30))))
        ax.axvline(x=d, color='#f5a623', linewidth=1, linestyle='--', alpha=0.7)
        ax.scatter([d], [p], color='#f5a623', zorder=5)
        ax.text(d * 1.05, p + 2, lbl, color='#f5a623', fontsize=9)

    ax.set_xlabel('Grain Diameter (mm)', color='white', fontsize=11)
    ax.set_ylabel('% Passing', color='white', fontsize=11)
    ax.set_title(f'Grain Size Distribution — {symbol}  (Cu={Cu:.1f}, Cc={Cc:.2f})', color='white', fontsize=12, pad=15)
    ax.set_ylim(0, 100)
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')
    ax.grid(True, which='both', color='#333', linestyle='--', alpha=0.5)
    ax.legend(facecolor='#1a1a2e', edgecolor='#555', labelcolor='white')
    plt.tight_layout()
    plt.savefig('grain_size_curve.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print('  Chart saved as grain_size_curve.png')


print('✓ Classification functions loaded')

In [ ]:
# ── Cell 3: Interactive Tool ──────────────────────────────────────────────────
# Adjust sliders then click Classify Soil

style  = {'description_width': '220px'}
layout = widgets.Layout(width='480px')

# ── Shared input ──────────────────────────────────────────────────────────────
w_p200 = widgets.FloatSlider(value=55, min=0, max=100, step=0.5,
                              description='% Passing #200 Sieve',
                              style=style, layout=layout, readout_format='.1f')

# ── Fine-grained inputs ───────────────────────────────────────────────────────
w_LL = widgets.FloatSlider(value=40, min=0,  max=110, step=1,
                            description='Liquid Limit LL (%)',
                            style=style, layout=layout)
w_PL = widgets.FloatSlider(value=20, min=0,  max=100, step=1,
                            description='Plastic Limit PL (%)',
                            style=style, layout=layout)

# ── Coarse-grained inputs ─────────────────────────────────────────────────────
w_gravel = widgets.FloatSlider(value=40, min=0, max=100, step=1,
                                description='% Gravel (retained #4)',
                                style=style, layout=layout)
w_Cu = widgets.FloatSlider(value=6, min=1, max=30, step=0.5,
                            description='Cu  (D60 / D10)',
                            style=style, layout=layout)
w_Cc = widgets.FloatSlider(value=2, min=0.1, max=5, step=0.1,
                            description='Cc  (D30² / D10·D60)',
                            style=style, layout=layout)
w_fLL = widgets.FloatSlider(value=30, min=0, max=110, step=1,
                             description='Fines LL (if fines ≥ 5%)',
                             style=style, layout=layout)
w_fPL = widgets.FloatSlider(value=15, min=0, max=100, step=1,
                             description='Fines PL (if fines ≥ 5%)',
                             style=style, layout=layout)

btn    = widgets.Button(description='  Classify Soil', button_style='success',
                        icon='flask', layout=widgets.Layout(width='200px', height='40px'))
output = widgets.Output()

# ── Dynamic panel: show relevant sliders based on % passing #200 ──────────────
fine_box   = widgets.VBox([widgets.HTML('<b>Fine-Grained Inputs</b>'), w_LL, w_PL])
coarse_box = widgets.VBox([widgets.HTML('<b>Coarse-Grained Inputs</b>'),
                            w_gravel, w_Cu, w_Cc,
                            widgets.HTML('<i style="color:gray">Atterberg limits for fines (optional if 5–12%, required if >12%)</i>'),
                            w_fLL, w_fPL])
panel = widgets.VBox([fine_box])

def update_panel(change):
    if w_p200.value > 50:
        panel.children = [fine_box]
    else:
        panel.children = [coarse_box]

w_p200.observe(update_panel, names='value')
update_panel(None)  # set initial state

# ── Classification callback ───────────────────────────────────────────────────
def on_classify(b):
    with output:
        clear_output(wait=True)
        pct_p200 = w_p200.value

        print('=' * 55)
        print('   USCS SOIL CLASSIFICATION TOOL')
        print('   Based on ASTM D2487')
        print('=' * 55)
        print(f'   % Passing #200 Sieve: {pct_p200:.1f}%')

        if pct_p200 > 50:
            LL = w_LL.value
            PL = w_PL.value
            PI = LL - PL
            if PI < 0:
                print('\n  ✗ Error: Plastic Limit cannot exceed Liquid Limit.')
                return
            print(f'   Soil Type : Fine-Grained')
            print(f'   LL = {LL:.0f}%,  PL = {PL:.0f}%,  PI = {PI:.1f}%')
            symbol, name, flag = classify_fine_grained(LL, PI)
        else:
            pct_fines  = pct_p200
            pct_gravel = w_gravel.value
            pct_sand   = max(0, 100 - pct_gravel - pct_fines)
            Cu, Cc     = w_Cu.value, w_Cc.value
            fines_LL   = w_fLL.value if pct_fines >= 5 else None
            fines_PL   = w_fPL.value if pct_fines >= 5 else None
            fines_PI   = (fines_LL - fines_PL) if (fines_LL is not None) else None
            print(f'   Soil Type : Coarse-Grained')
            print(f'   Gravel = {pct_gravel:.0f}%,  Sand = {pct_sand:.1f}%,  Fines = {pct_fines:.1f}%')
            print(f'   Cu = {Cu:.1f},  Cc = {Cc:.2f}')
            symbol, name, flag = classify_coarse_grained(
                pct_gravel, pct_sand, pct_fines, Cu, Cc, fines_LL, fines_PI)

        print('\n' + '─' * 55)
        if symbol:
            print(f'  USCS Symbol     : {symbol}')
            print(f'  Classification  : {name}')
        else:
            print(f'  ✗ {name}')
        if flag:
            print(f'  {flag}')
        print('─' * 55)

        if symbol:
            if pct_p200 > 50:
                plot_casagrande(w_LL.value, w_LL.value - w_PL.value, symbol)
            else:
                plot_grain_size(w_Cu.value, w_Cc.value, symbol)

btn.on_click(on_classify)

display(widgets.VBox([
    widgets.HTML('<h3 style="font-family:monospace">🪨 USCS Soil Classifier</h3>'),
    widgets.HTML('<hr>'),
    widgets.HTML('<b>Step 1 — Enter % passing #200 sieve to determine soil type</b>'),
    w_p200,
    widgets.HTML('<b>Step 2 — Fill in the inputs that appear below</b>'),
    panel,
    widgets.HTML('<br>'),
    btn,
    widgets.HTML('<br>'),
    output
]))